# CNM Analysis: dA RHIC (200 GeV)

**System:** dAu 200 GeV  
**Physics:** nPDF (EPPS21) × Energy Loss (Arleo-Peigne) × pT Broadening (Cronin)  
**Module:** `cnm_combine_fast` (Vectorized)


In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Add paths
HERE = Path(".").resolve()
sys.path.append(str(HERE.parent / "eloss_code"))
sys.path.append(str(HERE.parent / "npdf_code"))
sys.path.append(str(HERE.parent / "cnm_combine"))

from cnm_combine_fast import CNMCombineFast

# Plotting style
plt.style.use('seaborn-v0_8-paper')
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 12

OUTDIR = HERE / "outputs" / "calculated_cnm_results"
OUTDIR.mkdir(parents=True, exist_ok=True)
print(f"Saving outputs to {OUTDIR}")

## 1. Initialize CNM Combine (RHIC)

In [ ]:
cnm = CNMCombineFast.from_defaults(
    energy="200",
    family="charmonia",
    particle_state="avg"
)

print(f"Initialized {cnm.system} @ {cnm.energy} GeV")
print(f"Centrality bins: {cnm.cent_bins}")

In [ ]:
%%time
y_cent, labels, bands_y = cnm.cnm_vs_y()
print("Done y.")

In [ ]:
# Reusing Plotting Helper
def plot_bands(x, bands, labels, xlabel, ylabel="$R_{dAu}$", title=""):
    fig, ax = plt.subplots(figsize=(9, 6))
    colors = plt.cm.viridis(np.linspace(0, 0.9, len(labels)))
    color_map = {lab: col for lab, col in zip(labels, colors)}
    color_map["MB"] = "black"
    plot_order = list(labels) + ["MB"]
    comp_styles = {
        "cnm": ("-", 0.2),
        "npdf": ("--", 0.1),
        "eloss_broad": (":", 0.1)
    }
    for comp, (ls, alpha) in comp_styles.items():
        if comp not in bands: continue
        Dc, Dlo, Dhi = bands[comp]
        for lab in plot_order:
            c = color_map.get(lab, "gray")
            if comp == "cnm":
                ax.plot(x, Dc[lab], color=c, linestyle=ls, linewidth=2)
                ax.fill_between(x, Dlo[lab], Dhi[lab], color=c, alpha=alpha)
            else:
                ax.plot(x, Dc[lab], color=c, linestyle=ls, linewidth=1, alpha=0.7)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_ylim(0.0, 1.4)
    ax.axhline(1.0, color='k', linewidth=0.5)
    return fig

fig = plot_bands(y_cent, bands_y, labels, "$y_{CMS}$", title="RHIC dAu 200 GeV")
fig.savefig(OUTDIR / "06_d_RHIC_RpA_vs_y.png", dpi=150)

In [ ]:
# Vs pT
windows = cnm.y_windows
for (y0, y1, desc) in windows:
    pt, labels, bands = cnm.cnm_vs_pT(y_window=(y0, y1))
    fig = plot_bands(pt, bands, labels, "$p_T$ [GeV]", title=f"RHIC {desc}")
    fig.savefig(OUTDIR / f"06_d_RHIC_RpA_vs_pT_{y0}_{y1}.png", dpi=150)
    plt.close(fig)